# Notebook 16 — Per-Step Gradient Dynamics: How Does the Damage Happen?
**CS 590NN | Amogh | Apr 25 — temporal characterization of sequential edit damage**

## Genuinely novel angle

NB11 showed the optimizer step destroys A. NB15 showed it's the in-place modification. **But neither tells us HOW the damage unfolds during optimization.**

The editing literature treats edits as atomic — "before edit" vs. "after edit" measurements only. No paper I've found characterizes the *temporal dynamics* of how prior edits get destroyed during subsequent optimization. This notebook does.

## Method

For each pair (A, B):
1. Edit A normally → measure `p_A_postA`
2. Run B's optimizer **step-by-step**, measuring at every step:
   - `p_new_A_t` (A's installed knowledge at step t)
   - `p_new_B_t` (B's progress at step t)
   - `kl_neutral_t` (drift on the neutral set)
3. Plot trajectories per pair and aggregate.

## What we'd learn from each possible pattern

| Pattern | Interpretation |
|---|---|
| **Monotonic decay** | A degrades smoothly — damage is gradual erosion |
| **Sudden collapse** | A is fine for N steps then crashes at N+1 — discrete phase transition |
| **Oscillation** | A drops then partially recovers as KL anchor pushes back |
| **A drops before B installs** | Damage is faster than installation — the protection mechanism is fundamentally too slow |

Each of these has different implications for designing better protection. The literature has no characterization at all.

## Compute

~12 min A100. 8 pairs × ~15 steps each, with per-step measurement.

### 16.0 Install (run once, restarts)

In [ ]:
import subprocess, sys, os
def pip(args): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + args)
pip(['numpy==1.26.4'])
pip(['transformer-lens', 'transformers', 'datasets', 'accelerate', 'einops', 'jaxtyping'])
pip(['matplotlib', 'scipy'])
print('Restarting...'); os.kill(os.getpid(), 9)

### 16.1 Imports + auto-fetch v3

In [ ]:
import torch, json, requests, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
from scipy import stats
from transformer_lens import HookedTransformer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RESULTS_DIR = Path('results_nb16'); RESULTS_DIR.mkdir(exist_ok=True)
FIG_DIR = RESULTS_DIR / 'figures'; FIG_DIR.mkdir(exist_ok=True)
torch.manual_seed(42); random.seed(42); np.random.seed(42)
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    free, total = torch.cuda.mem_get_info()
    print(f'GPU: {torch.cuda.get_device_name(0)} ({free/1e9:.1f}/{total/1e9:.1f} GB free)')

REPO_RAW = 'https://raw.githubusercontent.com/rukmini-17/targeted-unlearning/main/circuit_pipeline/results'
V3_URL = f'{REPO_RAW}/sequential_editing_v3/sequential_edit_rows_v3_20260424_220951.json'
v3_data = requests.get(V3_URL, timeout=60).json()
v3_ok = [r for r in v3_data['rows'] if r.get('status') == 'ok']
seen = set()
PAIR_INDICES = []
for r in v3_ok:
    pid = r['pair_idx']
    if pid in seen: continue
    seen.add(pid)
    PAIR_INDICES.append((r['A_idx'], r['B_idx']))
    if len(PAIR_INDICES) >= 8: break
print(f'Selected {len(PAIR_INDICES)} pairs')

### 16.2 Load model + samples

In [ ]:
MODEL_NAME = 'Qwen/Qwen3-0.6B'
model = HookedTransformer.from_pretrained(
    MODEL_NAME,
    center_unembed=True, center_writing_weights=False,
    fold_ln=True, refactor_factored_attn_matrices=False, device=DEVICE,
)
model.set_use_attn_result(True)
model.eval()
model.tokenizer.pad_token = model.tokenizer.eos_token

@dataclass
class EditSample:
    idx: int; prompt: str; subject: str
    target_new: str; target_true: str

raw = requests.get('https://rome.baulab.info/data/dsets/counterfact.json', timeout=120).json()
def make_sample(idx):
    rr = raw[idx]['requested_rewrite']
    return EditSample(idx=idx, prompt=rr['prompt'].format(rr['subject']),
                      subject=rr['subject'],
                      target_new=' '+rr['target_new']['str'], target_true=' '+rr['target_true']['str'])

pairs = [{'A': make_sample(a), 'B': make_sample(b)} for a, b in PAIR_INDICES]
print(f'Loaded model + {len(pairs)} pairs')

### 16.3 ROME-trace + helpers

In [ ]:
def rome_trace_top_layers(model, sample, k=5):
    new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0].item()
    true_id = model.to_tokens(sample.target_true, prepend_bos=False)[0,0].item()
    tokens = model.to_tokens(sample.prompt)
    n_tok = tokens.shape[1]
    if n_tok <= 2: return list(range(min(k, model.cfg.n_layers)))
    subj_pos = list(range(1, n_tok-1))
    corrupt = tokens.clone()
    corrupt[0, subj_pos] = torch.randint(1000, model.cfg.d_vocab-1, (len(subj_pos),), device=tokens.device)
    with torch.no_grad():
        cl, cc = model.run_with_cache(tokens)
        kl, _ = model.run_with_cache(corrupt)
    clean_score = (cl[0,-1,true_id] - cl[0,-1,new_id]).item()
    corrupt_score = (kl[0,-1,true_id] - kl[0,-1,new_id]).item()
    eff = max(abs(clean_score - corrupt_score), 0.5)
    effects = []
    for L in range(model.cfg.n_layers):
        hn = f'blocks.{L}.hook_mlp_out'
        if hn not in cc: continue
        ca = cc[hn].clone()
        def hk(v, hook): return ca
        with torch.no_grad():
            patched = model.run_with_hooks(corrupt, fwd_hooks=[(hn, hk)])
        recovery = (patched[0,-1,true_id] - patched[0,-1,new_id]).item() - corrupt_score
        effects.append((L, abs(recovery)/eff))
    effects.sort(key=lambda x: -x[1])
    del cc; torch.cuda.empty_cache()
    return [L for L, _ in effects[:k]]

NEUTRAL = [
    'The sum of two and three is', 'Twelve divided by four equals',
    'The square root of nine is', 'Ten times ten equals',
    'The capital of Japan is', 'The largest ocean on Earth is the',
    'Mount Everest is located in the', 'The Amazon River flows through',
    'Water boils at one hundred degrees', 'The chemical symbol for gold is',
    'Plants produce oxygen through a process called', 'The Earth orbits the',
    'A week contains seven', 'The primary colors are red, blue, and',
    'Humans have two lungs and one', 'A triangle has three',
]

def cache_ref(model, prompts):
    cache = []
    model.eval()
    with torch.no_grad():
        for p in prompts:
            t = model.to_tokens(p)
            ref_lp = torch.log_softmax(model(t)[0,-1,:], dim=-1).detach().clone()
            cache.append((t, ref_lp))
    return cache

def kl_against(model, ref_cache):
    total = 0.0
    for t, ref_lp in ref_cache:
        edit_lp = torch.nn.functional.log_softmax(model(t)[0,-1,:], dim=-1)
        total = total + (ref_lp.exp() * (ref_lp - edit_lp)).sum()
    return total / len(ref_cache)

def measure_p_new(model, sample):
    model.eval()
    with torch.no_grad():
        new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0]
        return float(torch.softmax(model(model.to_tokens(sample.prompt))[0,-1,:], dim=-1)[new_id])

def measure_kl(model, ref_cache):
    model.eval()
    with torch.no_grad():
        return float(kl_against(model, ref_cache))

def restore(model, state):
    with torch.no_grad():
        for n, p in model.named_parameters(): p.copy_(state[n])
    torch.cuda.empty_cache()

def edit_normal(model, sample, mlp_layers, n_steps, lr, beta_kl, ref_cache):
    """Standard edit, no per-step recording — for installing A."""
    params = []
    for L in mlp_layers:
        params += [model.blocks[L].mlp.W_in, model.blocks[L].mlp.W_out]
    if not params:
        for L in range(model.cfg.n_layers):
            params += [model.blocks[L].mlp.W_in, model.blocks[L].mlp.W_out]
    for p in model.parameters(): p.requires_grad_(False)
    for p in params: p.requires_grad_(True)
    opt = torch.optim.Adam(params, lr=lr)
    new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0]
    true_id = model.to_tokens(sample.target_true, prepend_bos=False)[0,0]
    tokens = model.to_tokens(sample.prompt)
    for _ in range(n_steps):
        model.train()
        opt.zero_grad(set_to_none=True)
        logits = model(tokens)
        lp = torch.nn.functional.log_softmax(logits[0,-1,:], dim=-1)
        loss = -lp[new_id] + lp[true_id]
        if ref_cache and beta_kl > 0:
            loss = loss + beta_kl * kl_against(model, ref_cache)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
        opt.step()
    for p in model.parameters(): p.requires_grad_(True)
    torch.cuda.empty_cache()

print('Snapshotting weights...')
original_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
torch.cuda.empty_cache()
print('Ready.')

### 16.4 Per-step edit B with measurement after each step

In [ ]:
def edit_B_with_trajectory(model, sample_B, sample_A, mlp_layers, n_steps, lr, beta_kl, ref_cache):
    """
    Run edit B step-by-step, measuring A's state after each step.
    Returns trajectory: list of dicts {step, p_new_A, p_new_B, kl_neutral, loss}.
    """
    params = []
    for L in mlp_layers:
        params += [model.blocks[L].mlp.W_in, model.blocks[L].mlp.W_out]
    if not params:
        for L in range(model.cfg.n_layers):
            params += [model.blocks[L].mlp.W_in, model.blocks[L].mlp.W_out]
    for p in model.parameters(): p.requires_grad_(False)
    for p in params: p.requires_grad_(True)
    opt = torch.optim.Adam(params, lr=lr)
    new_id_B = model.to_tokens(sample_B.target_new, prepend_bos=False)[0,0]
    true_id_B = model.to_tokens(sample_B.target_true, prepend_bos=False)[0,0]
    tokens_B = model.to_tokens(sample_B.prompt)
    
    trajectory = []
    
    # Step 0: measurement before any optimizer step
    p_A_t = measure_p_new(model, sample_A)
    p_B_t = measure_p_new(model, sample_B)
    kl_t = measure_kl(model, ref_cache)
    trajectory.append({'step': 0, 'p_new_A': p_A_t, 'p_new_B': p_B_t, 'kl_neutral': kl_t, 'loss': None})
    
    for step in range(1, n_steps + 1):
        model.train()
        opt.zero_grad(set_to_none=True)
        logits = model(tokens_B)
        lp = torch.nn.functional.log_softmax(logits[0,-1,:], dim=-1)
        loss_target = -lp[new_id_B] + lp[true_id_B]
        if ref_cache and beta_kl > 0:
            loss_kl = beta_kl * kl_against(model, ref_cache)
            loss = loss_target + loss_kl
        else:
            loss = loss_target
        loss_val = float(loss.item())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
        opt.step()
        
        # Measure after the step
        p_A_t = measure_p_new(model, sample_A)
        p_B_t = measure_p_new(model, sample_B)
        kl_t = measure_kl(model, ref_cache)
        trajectory.append({'step': step, 'p_new_A': p_A_t, 'p_new_B': p_B_t, 'kl_neutral': kl_t, 'loss': loss_val})
    
    for p in model.parameters(): p.requires_grad_(True)
    torch.cuda.empty_cache()
    return trajectory

print('Per-step edit function ready.')

### 16.5 Run trajectories for all pairs

In [ ]:
N_STEPS_A = 5
N_STEPS_B = 15  # more steps to see the dynamics
BETA_KL = 0.1
LR = 5e-3

ref_cache = cache_ref(model, NEUTRAL)

all_samples = {s.idx: s for p in pairs for s in (p['A'], p['B'])}
print(f'ROME-trace for {len(all_samples)} samples...')
circuits = {idx: rome_trace_top_layers(model, s, k=5) for idx, s in all_samples.items()}
print('Done.')

ROWS_FILE = RESULTS_DIR / f'trajectories_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
all_trajectories = []
started = datetime.now()

for p_idx, p in enumerate(pairs):
    A, B = p['A'], p['B']
    try:
        # Reset and edit A normally
        restore(model, original_state)
        edit_normal(model, A, circuits[A.idx], n_steps=N_STEPS_A, lr=LR, beta_kl=BETA_KL, ref_cache=ref_cache)
        p_A_postA = measure_p_new(model, A)
        
        # Now run edit B step by step, measuring at each step
        traj = edit_B_with_trajectory(model, B, A, circuits[B.idx],
                                       n_steps=N_STEPS_B, lr=LR, beta_kl=BETA_KL, ref_cache=ref_cache)
        
        # Tag with metadata
        for t in traj:
            t['pair_idx'] = p_idx
            t['A_idx'] = A.idx
            t['B_idx'] = B.idx
            t['p_A_postA'] = p_A_postA
        all_trajectories.extend(traj)
        
        final = traj[-1]
        print(f'[{p_idx+1}/{len(pairs)}] pair {p_idx}: '
              f'p_A_postA={p_A_postA:.3f} → '
              f'final p_A={final["p_new_A"]:.3f}, p_B={final["p_new_B"]:.3f}, kl={final["kl_neutral"]:.3f}')
        
        with open(ROWS_FILE, 'w') as f:
            json.dump({'trajectories': all_trajectories}, f, indent=2)
    except Exception as e:
        print(f'[{p_idx+1}/{len(pairs)}] FAILED: {type(e).__name__}: {e}')
        torch.cuda.empty_cache()

elapsed = (datetime.now() - started).total_seconds() / 60
print(f'\nTotal runtime: {elapsed:.1f} min')
restore(model, original_state)

### 16.6 Characterize the dynamics

In [ ]:
traj_df = pd.DataFrame(all_trajectories)
print(f'Trajectory rows: {len(traj_df)}')
print(f'Pairs: {traj_df["pair_idx"].nunique()}')

# Per-step aggregate (mean ± std across pairs)
agg = traj_df.groupby('step').agg(
    p_A_mean=('p_new_A', 'mean'),
    p_A_std=('p_new_A', 'std'),
    p_A_median=('p_new_A', 'median'),
    p_B_mean=('p_new_B', 'mean'),
    p_B_std=('p_new_B', 'std'),
    kl_mean=('kl_neutral', 'mean'),
).round(3)
print('\n=== Per-step aggregate ===')
print(agg)

# Identify the "collapse step" per pair: first step where p_A drops by >50% of starting value
print('\n=== Per-pair collapse step ===')
collapse_steps = []
for p_idx in traj_df['pair_idx'].unique():
    sub = traj_df[traj_df['pair_idx']==p_idx].sort_values('step')
    p_A_0 = sub.iloc[0]['p_new_A']
    threshold = p_A_0 * 0.5
    collapsed = sub[sub['p_new_A'] < threshold]
    if len(collapsed) > 0:
        c_step = int(collapsed.iloc[0]['step'])
        collapse_steps.append({'pair_idx': p_idx, 'collapse_step': c_step, 'p_A_0': p_A_0})
        print(f'  pair {p_idx}: p_A starts at {p_A_0:.3f}, collapses below 50% at step {c_step}')
    else:
        collapse_steps.append({'pair_idx': p_idx, 'collapse_step': None, 'p_A_0': p_A_0})
        print(f'  pair {p_idx}: p_A starts at {p_A_0:.3f}, NEVER collapses (final = {sub.iloc[-1]["p_new_A"]:.3f})')

cs_df = pd.DataFrame(collapse_steps)
collapsed = cs_df[cs_df['collapse_step'].notna()]
print(f'\n  Collapsed: {len(collapsed)}/{len(cs_df)} pairs')
if len(collapsed) > 0:
    print(f'  Median collapse step: {collapsed["collapse_step"].median():.0f}')
    print(f'  Mean collapse step:   {collapsed["collapse_step"].mean():.1f}')

# Does B install before or after A collapses?
print('\n=== Does A collapse before, during, or after B installs? ===')
for p_idx in traj_df['pair_idx'].unique():
    sub = traj_df[traj_df['pair_idx']==p_idx].sort_values('step')
    # Step where p_B first exceeds 0.5
    b_install = sub[sub['p_new_B'] > 0.5]
    b_step = int(b_install.iloc[0]['step']) if len(b_install) > 0 else None
    a_collapse_row = cs_df[cs_df['pair_idx']==p_idx].iloc[0]
    a_step = a_collapse_row['collapse_step']
    if b_step is not None and a_step is not None:
        diff = a_step - b_step
        verdict = 'BEFORE B installs' if diff < 0 else ('AT same step as B' if diff == 0 else 'AFTER B installs')
        print(f'  pair {p_idx}: A collapses at step {a_step:.0f}, B installs at step {b_step}, A collapse {verdict}')
    else:
        print(f'  pair {p_idx}: A collapse step={a_step}, B install step={b_step}')

### 16.7 Money plots — temporal dynamics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left — per-pair p_A trajectories
ax = axes[0]
for p_idx in traj_df['pair_idx'].unique():
    sub = traj_df[traj_df['pair_idx']==p_idx].sort_values('step')
    ax.plot(sub['step'], sub['p_new_A'], '-o', alpha=0.6, ms=4, label=f'pair {p_idx}')
mean_traj = traj_df.groupby('step')['p_new_A'].mean()
ax.plot(mean_traj.index, mean_traj.values, 'k-', lw=3, label='MEAN', zorder=10)
ax.set_xlabel('B optimizer step')
ax.set_ylabel('p_new_A')
ax.set_title('A degradation over B optimization steps')
ax.grid(alpha=0.3); ax.set_ylim(-0.05, 1.05)
ax.legend(loc='center right', fontsize=8)

# Middle — A vs B trajectories on same axes (mean only)
ax = axes[1]
mean_A = traj_df.groupby('step')['p_new_A'].mean()
mean_B = traj_df.groupby('step')['p_new_B'].mean()
mean_kl = traj_df.groupby('step')['kl_neutral'].mean()
ax.plot(mean_A.index, mean_A.values, '-o', color='#cc3333', lw=2, ms=6, label='p_new_A (prior edit)')
ax.plot(mean_B.index, mean_B.values, '-o', color='#3366cc', lw=2, ms=6, label='p_new_B (new edit)')
ax2 = ax.twinx()
ax2.plot(mean_kl.index, mean_kl.values, '--s', color='#33aa33', lw=1.5, ms=5, label='KL neutral', alpha=0.7)
ax.set_xlabel('B optimizer step')
ax.set_ylabel('p_new_A / p_new_B')
ax2.set_ylabel('KL on neutral set', color='#33aa33')
ax.set_title('A collapses while B installs')
ax.grid(alpha=0.3); ax.set_ylim(-0.05, 1.05)
ax.legend(loc='upper left'); ax2.legend(loc='center right')

# Right — distribution of collapse steps
ax = axes[2]
cs_collapsed = cs_df[cs_df['collapse_step'].notna()]
if len(cs_collapsed) > 0:
    ax.hist(cs_collapsed['collapse_step'].astype(int), bins=range(0, N_STEPS_B+2),
            edgecolor='black', color='#cc6633', alpha=0.85)
    ax.axvline(cs_collapsed['collapse_step'].median(), color='red', ls='--',
               label=f'median = {cs_collapsed["collapse_step"].median():.0f}')
ax.set_xlabel('Step at which p_new_A drops below 50% of initial')
ax.set_ylabel('Number of pairs')
ax.set_title(f'Collapse step distribution ({len(cs_collapsed)}/{len(cs_df)} collapsed)')
ax.grid(alpha=0.3, axis='y'); ax.legend()

fig.tight_layout()
fig.savefig(FIG_DIR / 'fig1_temporal_dynamics.png', dpi=140)
plt.show()

### 16.8 Save and bundle

In [ ]:
ts = datetime.now().strftime('%Y%m%d_%H%M%S')

# Pattern characterization
mean_A = traj_df.groupby('step')['p_new_A'].mean().values
diffs = np.diff(mean_A)
biggest_drop_step = int(np.argmin(diffs)) + 1
biggest_drop = float(-diffs.min())

if biggest_drop > 0.4:
    pattern = 'sudden_collapse'
elif np.all(diffs <= 0.01) and (mean_A[0] - mean_A[-1]) > 0.3:
    pattern = 'monotonic_decay'
elif (diffs > 0).any() and (diffs < 0).any():
    pattern = 'oscillation'
else:
    pattern = 'gradual_decay'

summary = {
    'notebook': 'Notebook 16 — Per-Step Gradient Dynamics',
    'timestamp': ts, 'model': MODEL_NAME, 'n_pairs': len(pairs), 'n_steps': N_STEPS_B,
    'data_source': 'pair indices auto-fetched from github.com/rukmini-17/targeted-unlearning',
    'mean_p_A_per_step': mean_A.tolist(),
    'mean_p_B_per_step': traj_df.groupby('step')['p_new_B'].mean().values.tolist(),
    'biggest_drop_step': biggest_drop_step,
    'biggest_drop_magnitude': biggest_drop,
    'pattern_classification': pattern,
    'collapse_steps': cs_df.to_dict(orient='records'),
    'fraction_collapsed': float(cs_df['collapse_step'].notna().sum() / len(cs_df)),
}
with open(RESULTS_DIR / f'summary_nb16_{ts}.json', 'w') as f:
    json.dump(summary, f, indent=2)
traj_df.to_csv(RESULTS_DIR / f'trajectories_{ts}.csv', index=False)

print(json.dumps({k: v for k, v in summary.items() if k != 'collapse_steps'}, indent=2))
print(f'\nPattern classification: {pattern.upper()}')

import shutil
bundle = f'nb16_results_{ts}'
shutil.make_archive(bundle, 'zip', RESULTS_DIR)
from google.colab import files
files.download(f'{bundle}.zip')